#### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn

from pathlib import Path
from scipy import sparse

from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    VotingClassifier,
    StackingClassifier
)
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import lightgbm as lgb

#### Loading Processed Datasets and Feature Matrices

In [2]:
processed_dir = Path("../data/processed")
features_dir = Path("../data/features")
models_dir = Path("../models")

train_df = pd.read_csv(processed_dir / "train.csv")
test_df = pd.read_csv(processed_dir / "test.csv")

y_train = train_df["intent"]
y_test = test_df["intent"]

X_train_word = sparse.load_npz(features_dir / "X_train_word.npz")
X_test_word = sparse.load_npz(features_dir / "X_test_word.npz")

X_train_char = sparse.load_npz(features_dir / "X_train_char.npz")
X_test_char = sparse.load_npz(features_dir / "X_test_char.npz")

X_train_combined = sparse.load_npz(features_dir / "X_train_combined.npz")
X_test_combined = sparse.load_npz(features_dir / "X_test_combined.npz")

print(f"Training Samples: {len(y_train):,}")
print(f"Testing Samples: {len(y_test):,}")
print(f"Number of Intent Classes: {y_train.nunique()}")

print(f"\nWord TF-IDF: {X_train_word.shape}")
print(f"Character TF-IDF: {X_train_char.shape}")
print(f"Combined TF-IDF: {X_train_combined.shape}")

print(f"\nTraining Labels Missing: {y_train.isnull().sum()}")
print(f"Testing Labels Missing: {y_test.isnull().sum()}")

Training Samples: 17,210
Testing Samples: 4,303
Number of Intent Classes: 27

Word TF-IDF: (17210, 2749)
Character TF-IDF: (17210, 6190)
Combined TF-IDF: (17210, 8939)

Training Labels Missing: 0
Testing Labels Missing: 0


#### Observations
- **17,210 training samples and 4,303 testing samples** were successfully loaded, covering all 27 customer intent classes.
- **Three TF-IDF representations are available:** word-level (2,749 features), character-level (6,190 features), and combined (8,939 features).
- **Decision:** Model development and hyperparameter tuning will use training data exclusively, preserving the held-out test set for final evaluation.

#### 2. Experiment Tracking

#### Configuring MLflow Experiment Tracking

In [3]:
mlflow_db_path = (Path.cwd().parent / "mlflow.db").resolve()

mlflow.set_tracking_uri(f"sqlite:///{mlflow_db_path.as_posix()}")

experiment_name = "Customer Support Intent Intelligence"
mlflow.set_experiment(experiment_name)

print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment Name: {experiment_name}")
print(f"Experiment ID: {mlflow.get_experiment_by_name(experiment_name).experiment_id}")

2026/11/04 22:39:17 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/04 22:39:17 INFO mlflow.store.db.utils: Updating database tables
2026/11/04 22:39:19 INFO mlflow.tracking.fluent: Experiment with name 'Customer Support Intent Intelligence' does not exist. Creating a new experiment.


MLflow Tracking URI: sqlite:///D:/Projects/ML and Data Science/Customer Support Intent Intelligence/mlflow.db
Experiment Name: Customer Support Intent Intelligence
Experiment ID: 1


#### Observations
- **MLflow experiment tracking was initialized successfully** using a project-level SQLite database.
- **The experiment `Customer Support Intent Intelligence` was created with ID 1**, providing a dedicated environment for model comparisons.
- **Decision:** Model configurations, cross-validation metrics, and relevant artifacts will be recorded in MLflow to support reproducibility and model selection.

#### Configuring Stratified Cross-Validation

In [4]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro"
}

print(f"Cross-Validation Strategy: {cv.__class__.__name__}")
print(f"Number of Folds: {cv.n_splits}")
print(f"Shuffling Enabled: {cv.shuffle}")
print(f"Random State: {cv.random_state}")

print("\nEvaluation Metrics:")
for metric in scoring:
    print(f"- {metric}")

Cross-Validation Strategy: StratifiedKFold
Number of Folds: 5
Shuffling Enabled: True
Random State: 42

Evaluation Metrics:
- accuracy
- f1_macro
- f1_weighted
- precision_macro
- recall_macro


#### Observations
- **5-fold stratified cross-validation** is configured with shuffling and a fixed random state of 42 for reproducibility.
- **Five evaluation metrics** will be tracked, including accuracy, macro F1-score, weighted F1-score, macro precision, and macro recall.
- **Decision:** Macro F1-score will serve as the primary model-selection metric to account for performance differences across the 27 imbalanced intent classes.

#### 3. Baseline Modeling

#### Evaluating the Dummy Classifier Baseline

In [5]:
dummy_model = DummyClassifier(
    strategy="most_frequent",
    random_state=42
)

dummy_cv_results = cross_validate(
    dummy_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

dummy_metrics = {
    metric: dummy_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Dummy Classifier - Baseline"):
    mlflow.log_param("model", "DummyClassifier")
    mlflow.log_param("strategy", "most_frequent")
    mlflow.log_param("feature_representation", "word_tfidf")
    mlflow.log_param("cv_folds", 5)

    for metric, value in dummy_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

dummy_results_df = pd.DataFrame(
    [dummy_metrics],
    index=["Dummy Classifier"]
).round(4)

display(dummy_results_df)

,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
Dummy Classifier,0.2029,0.0125,0.0685,0.0075,0.037


#### Observations
- **The Dummy Classifier achieves 20.29% accuracy**, reflecting the proportion of the majority intent class, `payment_issue`.
- **Macro F1-score is only 0.0125**, with macro recall of 0.0370, indicating extremely poor performance across the 27 intent classes.
- **Key Finding:** Majority-class predictions provide an inadequate classification strategy, reinforcing the importance of macro F1-score for evaluating subsequent models.

## 4. Probabilistic Model Development

#### Evaluating Multinomial Naive Bayes

In [6]:
mnb_model = MultinomialNB(alpha=1.0)

mnb_cv_results = cross_validate(
    mnb_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

mnb_metrics = {
    metric: mnb_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Multinomial Naive Bayes - Word TF-IDF"):
    mlflow.log_param("model", "MultinomialNB")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_param("feature_representation", "word_tfidf")
    mlflow.log_param("cv_folds", 5)

    for metric, value in mnb_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

mnb_results_df = pd.DataFrame(
    [mnb_metrics],
    index=["Multinomial Naive Bayes"]
).round(4)

display(mnb_results_df)

,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
Multinomial Naive Bayes,0.9794,0.8921,0.9767,0.9511,0.8699


## 5. Model Performance Tracking

#### Initializing the Model Comparison Results

In [7]:
model_results = []

for model_name, metrics in [
    ("Dummy Classifier", dummy_metrics),
    ("Multinomial Naive Bayes", mnb_metrics)
]:
    model_results.append({
        "Model": model_name,
        "Feature Representation": "Word TF-IDF",
        **metrics
    })

results_df = pd.DataFrame(model_results)

results_path = processed_dir / "model_comparison_results.csv"
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699



Results saved to: ..\data\processed\model_comparison_results.csv


#### Evaluating Complement Naive Bayes (Word TF-IDF)

In [8]:
cnb_model = ComplementNB(alpha=1.0)

cnb_cv_results = cross_validate(
    cnb_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

cnb_metrics = {
    metric: cnb_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Complement Naive Bayes - Word TF-IDF"):
    mlflow.log_param("model", "ComplementNB")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_param("feature_representation", "word_tfidf")
    mlflow.log_param("cv_folds", 5)

    for metric, value in cnb_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Complement Naive Bayes",
    "Feature Representation": "Word TF-IDF",
    **cnb_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699
2,Complement Naive Bayes,Word TF-IDF,0.9813,0.9339,0.9807,0.9727,0.9177



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Complement Naive Bayes achieves 98.13% accuracy** and a weighted F1-score of 0.9807, outperforming Multinomial Naive Bayes.
- **Macro F1-score improves from 0.8921 to 0.9339**, while macro recall increases from 0.8699 to 0.9177, indicating better overall performance across intent classes.
- **Key Finding:** Complement Naive Bayes is the strongest model evaluated so far, demonstrating its effectiveness for imbalanced customer intent classification.

## 6. Linear Model Development

#### Evaluating Logistic Regression (Word TF-IDF)

In [9]:
lr_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver="lbfgs",
    random_state=42
)

lr_cv_results = cross_validate(
    lr_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

lr_metrics = {
    metric: lr_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Logistic Regression - Word TF-IDF"):
    mlflow.log_params({
        "model": "LogisticRegression",
        "C": 1.0,
        "max_iter": 1000,
        "solver": "lbfgs",
        "feature_representation": "word_tfidf",
        "cv_folds": 5
    })

    for metric, value in lr_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Logistic Regression",
    "Feature Representation": "Word TF-IDF",
    **lr_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699
2,Complement Naive Bayes,Word TF-IDF,0.9813,0.9339,0.9807,0.9727,0.9177
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Logistic Regression achieves 99.57% accuracy** and a weighted F1-score of 0.9956, outperforming both Naive Bayes classifiers.
- **Macro F1-score reaches 0.9859**, with macro precision of 0.9960 and macro recall of 0.9778, demonstrating strong performance across the 27 intent classes.
- **Key Finding:** Logistic Regression is currently the strongest model, establishing a high benchmark for subsequent classifiers.

#### Evaluating Linear SVM (Word TF-IDF)

In [10]:
svm_model = LinearSVC(
    C=1.0,
    max_iter=2000,
    random_state=42
)

svm_cv_results = cross_validate(
    svm_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

svm_metrics = {
    metric: svm_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Linear SVM - Word TF-IDF"):
    mlflow.log_params({
        "model": "LinearSVC",
        "C": 1.0,
        "max_iter": 2000,
        "feature_representation": "word_tfidf",
        "cv_folds": 5
    })

    for metric, value in svm_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Linear SVM",
    "Feature Representation": "Word TF-IDF",
    **svm_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699
2,Complement Naive Bayes,Word TF-IDF,0.9813,0.9339,0.9807,0.9727,0.9177
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778
4,Linear SVM,Word TF-IDF,0.9975,0.9924,0.9975,0.9954,0.9904



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Linear SVM achieves 99.75% accuracy** and a weighted F1-score of 0.9975, outperforming all previously evaluated models.
- **Macro F1-score improves to 0.9924**, with macro precision of 0.9954 and macro recall of 0.9904, indicating consistently strong classification performance across intent classes.
- **Key Finding:** Linear SVM is currently the strongest classifier, demonstrating the effectiveness of linear models on high-dimensional Word TF-IDF features.

#### Evaluating SGD Classifier (Word TF-IDF)

In [11]:
sgd_model = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=0.0001,
    max_iter=1000,
    tol=1e-3,
    random_state=42
)

sgd_cv_results = cross_validate(
    sgd_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

sgd_metrics = {
    metric: sgd_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="SGD Classifier - Word TF-IDF"):
    mlflow.log_params({
        "model": "SGDClassifier",
        "loss": "log_loss",
        "penalty": "l2",
        "alpha": 0.0001,
        "max_iter": 1000,
        "feature_representation": "word_tfidf",
        "cv_folds": 5
    })

    for metric, value in sgd_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "SGD Classifier",
    "Feature Representation": "Word TF-IDF",
    **sgd_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699
2,Complement Naive Bayes,Word TF-IDF,0.9813,0.9339,0.9807,0.9727,0.9177
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778
4,Linear SVM,Word TF-IDF,0.9975,0.9924,0.9975,0.9954,0.9904
5,SGD Classifier,Word TF-IDF,0.9944,0.9823,0.9943,0.9954,0.9719



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **SGD Classifier achieves 99.44% accuracy** and a weighted F1-score of 0.9943, demonstrating strong performance with Word TF-IDF features.
- **Macro F1-score reaches 0.9823**, slightly below Logistic Regression (0.9859) and Linear SVM (0.9924).
- **Key Finding:** Linear SVM remains the strongest classifier, while SGD Classifier provides another competitive linear modeling approach.

## 7. Tree-Based Ensemble Model Development

#### Evaluating Random Forest (Word TF-IDF)

In [12]:
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=1
)

rf_cv_results = cross_validate(
    rf_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=2
)

rf_metrics = {
    metric: rf_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Random Forest - Word TF-IDF"):
    mlflow.log_params({
        "model": "RandomForestClassifier",
        "n_estimators": 150,
        "max_features": "sqrt",
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "feature_representation": "word_tfidf",
        "cv_folds": 5
    })

    for metric, value in rf_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Random Forest",
    "Feature Representation": "Word TF-IDF",
    **rf_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699
2,Complement Naive Bayes,Word TF-IDF,0.9813,0.9339,0.9807,0.9727,0.9177
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778
4,Linear SVM,Word TF-IDF,0.9975,0.9924,0.9975,0.9954,0.9904
5,SGD Classifier,Word TF-IDF,0.9944,0.9823,0.9943,0.9954,0.9719
6,Random Forest,Word TF-IDF,0.9935,0.9869,0.9935,0.9906,0.9841



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Random Forest achieves 99.35% accuracy** and a weighted F1-score of 0.9935, demonstrating strong performance with Word TF-IDF features.
- **Macro F1-score reaches 0.9869**, outperforming Logistic Regression (0.9859) and SGD Classifier (0.9823), although remaining below Linear SVM (0.9924).
- **Key Finding:** Random Forest provides competitive nonlinear classification performance, while Linear SVM remains the strongest overall model.

#### Evaluating Extra Trees (Word TF-IDF)

In [13]:
et_model = ExtraTreesClassifier(
    n_estimators=150,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=1
)

et_cv_results = cross_validate(
    et_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=2
)

et_metrics = {
    metric: et_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Extra Trees - Word TF-IDF"):
    mlflow.log_params({
        "model": "ExtraTreesClassifier",
        "n_estimators": 150,
        "max_features": "sqrt",
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "feature_representation": "word_tfidf",
        "cv_folds": 5
    })

    for metric, value in et_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Extra Trees",
    "Feature Representation": "Word TF-IDF",
    **et_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699
2,Complement Naive Bayes,Word TF-IDF,0.9813,0.9339,0.9807,0.9727,0.9177
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778
4,Linear SVM,Word TF-IDF,0.9975,0.9924,0.9975,0.9954,0.9904
5,SGD Classifier,Word TF-IDF,0.9944,0.9823,0.9943,0.9954,0.9719
6,Random Forest,Word TF-IDF,0.9935,0.9869,0.9935,0.9906,0.9841
7,Extra Trees,Word TF-IDF,0.9950,0.9869,0.9950,0.9901,0.9852



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Extra Trees achieves 99.50% accuracy** and a weighted F1-score of 0.9950, slightly outperforming Random Forest on both metrics.
- **Macro F1-score reaches 0.9869**, matching Random Forest, while macro recall improves marginally from 0.9841 to 0.9852.
- **Key Finding:** Extra Trees is a competitive tree-based classifier, although Linear SVM remains the strongest overall model with a macro F1-score of 0.9924.

#### Evaluating LightGBM (Word TF-IDF)

In [14]:
lgbm_model = lgb.LGBMClassifier(
    objective="multiclass",
    n_estimators=150,
    learning_rate=0.1,
    num_leaves=15,
    max_depth=7,
    min_child_samples=20,
    colsample_bytree=0.8,
    subsample=0.8,
    subsample_freq=1,
    verbosity=-1,
    random_state=42,
    n_jobs=2
)

lgbm_cv_results = cross_validate(
    lgbm_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

lgbm_metrics = {
    metric: lgbm_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="LightGBM - Word TF-IDF"):
    mlflow.log_params({
        "model": "LGBMClassifier",
        "n_estimators": 150,
        "learning_rate": 0.1,
        "num_leaves": 15,
        "max_depth": 7,
        "min_child_samples": 20,
        "feature_representation": "word_tfidf",
        "cv_folds": 5
    })

    for metric, value in lgbm_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "LightGBM",
    "Feature Representation": "Word TF-IDF",
    **lgbm_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699
2,Complement Naive Bayes,Word TF-IDF,0.9813,0.9339,0.9807,0.9727,0.9177
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778
4,Linear SVM,Word TF-IDF,0.9975,0.9924,0.9975,0.9954,0.9904
5,SGD Classifier,Word TF-IDF,0.9944,0.9823,0.9943,0.9954,0.9719
6,Random Forest,Word TF-IDF,0.9935,0.9869,0.9935,0.9906,0.9841
7,Extra Trees,Word TF-IDF,0.9950,0.9869,0.9950,0.9901,0.9852
8,LightGBM,Word TF-IDF,0.9913,0.9761,0.9912,0.9815,0.9724



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **LightGBM achieves 99.13% accuracy** and a weighted F1-score of 0.9912, demonstrating strong multiclass classification performance.
- **Macro F1-score reaches 0.9761**, falling below Random Forest (0.9869), Extra Trees (0.9869), and Linear SVM (0.9924).
- **Key Finding:** Gradient boosting performs effectively on Word TF-IDF features, although Linear SVM remains the strongest overall classifier.

## 8. Neural Network Model Development

#### Evaluating Multi-Layer Perceptron (Word TF-IDF)

In [15]:
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128,),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=128,
    learning_rate_init=0.001,
    max_iter=50,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5,
    random_state=42
)

mlp_cv_results = cross_validate(
    mlp_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

mlp_metrics = {
    metric: mlp_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="MLP Classifier - Word TF-IDF"):
    mlflow.log_params({
        "model": "MLPClassifier",
        "hidden_layer_sizes": "(128,)",
        "activation": "relu",
        "solver": "adam",
        "alpha": 0.0001,
        "batch_size": 128,
        "max_iter": 50,
        "early_stopping": True,
        "feature_representation": "word_tfidf",
        "cv_folds": 5
    })

    for metric, value in mlp_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "MLP Classifier",
    "Feature Representation": "Word TF-IDF",
    **mlp_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699
2,Complement Naive Bayes,Word TF-IDF,0.9813,0.9339,0.9807,0.9727,0.9177
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778
4,Linear SVM,Word TF-IDF,0.9975,0.9924,0.9975,0.9954,0.9904
5,SGD Classifier,Word TF-IDF,0.9944,0.9823,0.9943,0.9954,0.9719
6,Random Forest,Word TF-IDF,0.9935,0.9869,0.9935,0.9906,0.9841
7,Extra Trees,Word TF-IDF,0.9950,0.9869,0.9950,0.9901,0.9852
8,LightGBM,Word TF-IDF,0.9913,0.9761,0.9912,0.9815,0.9724
9,MLP Classifier,Word TF-IDF,0.9966,0.9892,0.9965,0.9948,0.9851



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **MLP Classifier achieves 99.66% accuracy** and a weighted F1-score of 0.9965, demonstrating excellent performance with Word TF-IDF features.
- **Macro F1-score reaches 0.9892**, outperforming Logistic Regression, Random Forest, and Extra Trees, while remaining slightly below Linear SVM (0.9924).
- **Key Finding:** MLP Classifier is the second-best model evaluated so far, highlighting the effectiveness of a lightweight neural network for customer intent classification.

## 9. Feature Representation Comparison

#### Evaluating Linear SVM (Character TF-IDF)

In [16]:
svm_char_model = LinearSVC(
    C=1.0,
    max_iter=2000,
    random_state=42
)

svm_char_cv_results = cross_validate(
    svm_char_model,
    X_train_char,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

svm_char_metrics = {
    metric: svm_char_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Linear SVM - Character TF-IDF"):
    mlflow.log_params({
        "model": "LinearSVC",
        "C": 1.0,
        "max_iter": 2000,
        "feature_representation": "character_tfidf",
        "cv_folds": 5
    })

    for metric, value in svm_char_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Linear SVM",
    "Feature Representation": "Character TF-IDF",
    **svm_char_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(
    results_df[
        results_df["Model"] == "Linear SVM"
    ].round(4)
)

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
4,Linear SVM,Word TF-IDF,0.9975,0.9924,0.9975,0.9954,0.9904
10,Linear SVM,Character TF-IDF,0.9995,0.9978,0.9995,0.9987,0.9971



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Linear SVM with Character TF-IDF achieves 99.95% accuracy** and a macro F1-score of 0.9978, outperforming the Word TF-IDF representation.
- **Macro recall improves from 0.9904 to 0.9971**, indicating stronger classification performance across the 27 intent classes.
- **Key Finding:** Character-level n-grams provide highly effective predictive features, making Character TF-IDF the strongest representation evaluated so far.

#### Evaluating Linear SVM (Combined TF-IDF)

In [17]:
svm_combined_model = LinearSVC(
    C=1.0,
    max_iter=2000,
    random_state=42
)

svm_combined_cv_results = cross_validate(
    svm_combined_model,
    X_train_combined,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

svm_combined_metrics = {
    metric: svm_combined_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Linear SVM - Combined TF-IDF"):
    mlflow.log_params({
        "model": "LinearSVC",
        "C": 1.0,
        "max_iter": 2000,
        "feature_representation": "combined_tfidf",
        "cv_folds": 5
    })

    for metric, value in svm_combined_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Linear SVM",
    "Feature Representation": "Combined TF-IDF",
    **svm_combined_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(
    results_df[
        results_df["Model"] == "Linear SVM"
    ].round(4)
)

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
4,Linear SVM,Word TF-IDF,0.9975,0.9924,0.9975,0.9954,0.9904
10,Linear SVM,Character TF-IDF,0.9995,0.9978,0.9995,0.9987,0.9971
11,Linear SVM,Combined TF-IDF,0.9995,0.9980,0.9995,0.9989,0.9974



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Combined TF-IDF achieves 99.95% accuracy** and a macro F1-score of 0.9980, delivering the strongest Linear SVM performance.
- **Character TF-IDF performs nearly identically**, with a macro F1-score of 0.9978, while Word TF-IDF achieves 0.9924.
- **Key Finding:** Combining word-level and character-level features provides a marginal improvement, making Combined TF-IDF the leading representation for Linear SVM.

#### Evaluating Logistic Regression (Character TF-IDF)

In [18]:
lr_char_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver="lbfgs",
    random_state=42
)

lr_char_cv_results = cross_validate(
    lr_char_model,
    X_train_char,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

lr_char_metrics = {
    metric: lr_char_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Logistic Regression - Character TF-IDF"):
    mlflow.log_params({
        "model": "LogisticRegression",
        "C": 1.0,
        "max_iter": 1000,
        "solver": "lbfgs",
        "feature_representation": "character_tfidf",
        "cv_folds": 5
    })

    for metric, value in lr_char_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Logistic Regression",
    "Feature Representation": "Character TF-IDF",
    **lr_char_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(
    results_df[
        results_df["Model"] == "Logistic Regression"
    ].round(4)
)

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778
12,Logistic Regression,Character TF-IDF,0.9985,0.9950,0.9985,0.9976,0.9930



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Logistic Regression with Character TF-IDF achieves 99.85% accuracy** and a macro F1-score of 0.9950, outperforming its Word TF-IDF counterpart.
- **Macro recall improves from 0.9778 to 0.9930**, indicating better performance across the 27 intent classes.
- **Key Finding:** Character-level features substantially improve Logistic Regression, reinforcing their effectiveness for customer intent classification.

#### Evaluating Logistic Regression (Combined TF-IDF)

In [19]:
lr_combined_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver="lbfgs",
    random_state=42
)

lr_combined_cv_results = cross_validate(
    lr_combined_model,
    X_train_combined,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

lr_combined_metrics = {
    metric: lr_combined_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Logistic Regression - Combined TF-IDF"):
    mlflow.log_params({
        "model": "LogisticRegression",
        "C": 1.0,
        "max_iter": 1000,
        "solver": "lbfgs",
        "feature_representation": "combined_tfidf",
        "cv_folds": 5
    })

    for metric, value in lr_combined_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Logistic Regression",
    "Feature Representation": "Combined TF-IDF",
    **lr_combined_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(
    results_df[
        results_df["Model"] == "Logistic Regression"
    ].round(4)
)

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778
12,Logistic Regression,Character TF-IDF,0.9985,0.9950,0.9985,0.9976,0.9930
13,Logistic Regression,Combined TF-IDF,0.9987,0.9939,0.9987,0.9976,0.9911



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Combined TF-IDF achieves the highest accuracy of 99.87%**, marginally outperforming Character TF-IDF (99.85%) and Word TF-IDF (99.57%).
- **Character TF-IDF achieves the highest macro F1-score of 0.9950** and macro recall of 0.9930, demonstrating stronger balanced performance across intent classes.
- **Key Finding:** Character TF-IDF is the preferred representation for Logistic Regression based on macro F1-score, while Combined TF-IDF offers slightly higher overall accuracy.

#### Evaluating MLP Classifier (Character TF-IDF)

In [20]:
mlp_char_model = MLPClassifier(
    hidden_layer_sizes=(128,),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=128,
    learning_rate_init=0.001,
    max_iter=50,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5,
    random_state=42
)

mlp_char_cv_results = cross_validate(
    mlp_char_model,
    X_train_char,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

mlp_char_metrics = {
    metric: mlp_char_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="MLP Classifier - Character TF-IDF"):
    mlflow.log_params({
        "model": "MLPClassifier",
        "hidden_layer_sizes": "(128,)",
        "activation": "relu",
        "solver": "adam",
        "alpha": 0.0001,
        "batch_size": 128,
        "max_iter": 50,
        "early_stopping": True,
        "feature_representation": "character_tfidf",
        "cv_folds": 5
    })

    for metric, value in mlp_char_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "MLP Classifier",
    "Feature Representation": "Character TF-IDF",
    **mlp_char_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(
    results_df[
        results_df["Model"] == "MLP Classifier"
    ].round(4)
)

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
9,MLP Classifier,Word TF-IDF,0.9966,0.9892,0.9965,0.9948,0.9851
14,MLP Classifier,Character TF-IDF,0.9983,0.9952,0.9982,0.9972,0.9938



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **MLP Classifier with Character TF-IDF achieves 99.83% accuracy** and a weighted F1-score of 0.9982, outperforming its Word TF-IDF counterpart.
- **Macro F1-score improves from 0.9892 to 0.9952**, while macro recall increases from 0.9851 to 0.9938.
- **Key Finding:** Character-level features substantially improve MLP performance, making Character TF-IDF the preferred representation evaluated so far.